In [4]:
# Requires transformers>=4.51.0

import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

# Each query must come with a one-sentence instruction that describes the task
task = 'Given a web search query, retrieve relevant passages that answer the query'

queries = [
    get_detailed_instruct(task, 'What is the capital of China?'),
    get_detailed_instruct(task, 'Explain gravity')
]
# No need to add instruction for retrieval documents
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
]
input_texts = queries + documents

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B', padding_side='left')
model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B')

# We recommend enabling flash_attention_2 for better acceleration and memory saving.
# model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B', attn_implementation="flash_attention_2", torch_dtype=torch.float16).cuda()

max_length = 8192

# Tokenize the input texts
batch_dict = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)
batch_dict.to(model.device)
outputs = model(**batch_dict)
embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T)
print(scores.tolist())
# [[0.7645568251609802, 0.14142508804798126], [0.13549736142158508, 0.5999549627304077]]


[[0.7645564675331116, 0.1414250135421753], [0.13549752533435822, 0.5999548435211182]]


In [1]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
print("config keys:", cfg.__dict__.keys())
print("num_hidden_layers:", getattr(cfg, "num_hidden_layers", None))
print("n_layer:", getattr(cfg, "n_layer", None))
print("hidden size =", getattr(cfg, "hidden_size"))
print("architectures:", getattr(cfg, "architectures"))
print("model type=", getattr(cfg, "model_type"))
print("num attention:", getattr(cfg, "num_attention_heads"))
print(getattr(cfg, "num_key_value_heads"))

config keys: dict_keys(['vocab_size', 'max_position_embeddings', 'hidden_size', 'intermediate_size', 'num_hidden_layers', 'num_attention_heads', 'use_sliding_window', 'sliding_window', 'max_window_layers', 'num_key_value_heads', 'head_dim', 'hidden_act', 'initializer_range', 'rms_norm_eps', 'use_cache', 'rope_theta', 'rope_scaling', 'attention_bias', 'attention_dropout', 'layer_types', 'return_dict', 'output_hidden_states', 'torchscript', 'dtype', '_output_attentions', 'pruned_heads', 'tie_word_embeddings', 'chunk_size_feed_forward', 'is_encoder_decoder', 'is_decoder', 'cross_attention_hidden_size', 'add_cross_attention', 'tie_encoder_decoder', 'architectures', 'finetuning_task', 'id2label', 'label2id', 'task_specific_params', 'problem_type', 'tokenizer_class', 'prefix', 'bos_token_id', 'pad_token_id', 'eos_token_id', 'sep_token_id', 'decoder_start_token_id', 'max_length', 'min_length', 'do_sample', 'early_stopping', 'num_beams', 'temperature', 'top_k', 'top_p', 'typical_p', 'repetitio

In [2]:
from transformers import AutoModelForCausalLM

m = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-Embedding-0.6B")

layer0 = m.model.layers[0]
attn = layer0.self_attn

print("num_attention_heads:", m.config.num_attention_heads)
print("num_key_value_heads:", m.config.num_key_value_heads)

# Common projection names in Qwen/Llama-like implementations:
for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
    if hasattr(attn, name):
        w = getattr(attn, name).weight
        print(name, "weight shape:", tuple(w.shape))


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

num_attention_heads: 16
num_key_value_heads: 8
q_proj weight shape: (2048, 1024)
k_proj weight shape: (1024, 1024)
v_proj weight shape: (1024, 1024)
o_proj weight shape: (1024, 2048)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen3-Embedding-0.6B"

tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name,
    attn_implementation="eager",  # important for returning attention weights
    torch_dtype="auto",
    device_map="auto",
)

text = "Inspect attention orientation per layer."
inputs = tok(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model(**inputs, output_attentions=True, return_dict=True)

attns = out.attentions  # tuple: one tensor per layer
print("layers:", len(attns), "layer0 shape:", attns[0].shape)

def orientation_stats(A_2d: torch.Tensor):
    # A_2d: (S, S) attention weights, rows sum to ~1
    S = A_2d.size(0)
    upper = A_2d.triu(1).sum()   # attends to future tokens
    lower = A_2d.tril(-1).sum()  # attends to past tokens
    diag  = A_2d.diag().sum()
    total = A_2d.sum() + 1e-12
    return (lower/total).item(), (diag/total).item(), (upper/total).item()

for l, A in enumerate(attns):
    # A: (B, H, S, S). Average heads for a simple summary.
    A_mean = A[0].mean(dim=0)  # (S, S)
    lower_frac, diag_frac, upper_frac = orientation_stats(A_mean)

    # Simple rule of thumb:
    # - causal: upper_frac ~ 0
    # - reversed: lower_frac ~ 0
    # - bidirectional: both upper_frac and lower_frac are non-trivial
    if upper_frac < 1e-4 and lower_frac > 1e-2:
        label = "causal"
    elif lower_frac < 1e-4 and upper_frac > 1e-2:
        label = "reversed"
    else:
        label = "bidirectional/mixed"

    print(f"layer {l:02d}: {label:16s} lower={lower_frac:.4f} diag={diag_frac:.4f} upper={upper_frac:.4f}")


layers: 28 layer0 shape: torch.Size([1, 16, 7, 7])
layer 00: causal           lower=0.2988 diag=0.6992 upper=0.0000
layer 01: causal           lower=0.5391 diag=0.4629 upper=0.0000
layer 02: causal           lower=0.5039 diag=0.4980 upper=0.0000
layer 03: causal           lower=0.7852 diag=0.2139 upper=0.0000
layer 04: causal           lower=0.7383 diag=0.2637 upper=0.0000
layer 05: causal           lower=0.7500 diag=0.2520 upper=0.0000
layer 06: causal           lower=0.7188 diag=0.2793 upper=0.0000
layer 07: causal           lower=0.7188 diag=0.2812 upper=0.0000
layer 08: causal           lower=0.7383 diag=0.2617 upper=0.0000
layer 09: causal           lower=0.6992 diag=0.3008 upper=0.0000
layer 10: causal           lower=0.7188 diag=0.2812 upper=0.0000
layer 11: causal           lower=0.6797 diag=0.3223 upper=0.0000
layer 12: causal           lower=0.7773 diag=0.2207 upper=0.0000
layer 13: causal           lower=0.7305 diag=0.2695 upper=0.0000
layer 14: causal           lower=0.7305

In [5]:
import torch
from transformers import AutoModel, AutoTokenizer
def hack_qwen3_last_layer_bidirectional(model):
    """
    Manually hacks a Qwen3 (or Qwen2) model to make the last layer bidirectional.
    """
    # 1. Identify the last layer
    last_layer_idx = len(model.layers) - 1
    last_layer = model.layers[last_layer_idx]
    
    # 2. Hack the attention attribute
    # In many implementations, setting is_causal=False tells the attention function
    # (like SDPA) to use the provided mask instead of an internal causal one.
    if hasattr(last_layer.self_attn, 'is_causal'):
        last_layer.self_attn.is_causal = False
        print(f"Layer {last_layer_idx} is_causal set to False")
    # 3. Monkey-patch the forward pass to handle different masks per layer
    original_forward = model.forward
    def hacked_forward(
        input_ids=None,
        attention_mask=None,
        **kwargs
    ):
        # If no attention_mask is provided, create a default one (all ones)
        if attention_mask is None:
            batch_size, seq_len = input_ids.shape
            attention_mask = torch.ones((batch_size, seq_len), device=input_ids.device)
        # The trick is that we need to pass a CAUSAL mask to layers 0 to N-1
        # and a NON-CAUSAL (bidirectional) mask to layer N.
        
        # However, standard Transformers models usually take one mask and apply it to all.
        # To truly hack it without rewriting the whole loop, we can intercept the call
        # or modify the mask dynamically.
        
        # A simpler way for "testing purposes" is to manually iterate through layers:
        
        # Get embeddings
        hidden_states = model.embed_tokens(input_ids)
        
        # Prepare masks
        # Standard causal mask (4D: [batch, 1, seq, seq])
        causal_mask = model._update_causal_mask(attention_mask, hidden_states, None, None, False)
        
        # Bidirectional mask (just padding mask expanded to 4D)
        bidir_mask = attention_mask[:, None, None, :].to(hidden_states.dtype)
        bidir_mask = (1.0 - bidir_mask) * torch.finfo(hidden_states.dtype).min
        
        # Loop through layers
        for i, layer in enumerate(model.layers):
            # Use bidir_mask for the last layer, causal_mask for others
            current_mask = bidir_mask if i == last_layer_idx else causal_mask
            
            layer_outputs = layer(
                hidden_states,
                attention_mask=current_mask,
                **kwargs
            )
            hidden_states = layer_outputs[0]
            
        # Final norm
        hidden_states = model.norm(hidden_states)
        return hidden_states
    # Replace the forward method
    # Note: In a real scenario, you'd want to be more careful with the signature
    model.forward = hacked_forward
    return model
# Example usage:
# model_name = "Qwen/Qwen2.5-0.5B" # Or Qwen3-Embedding
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name)
# model = hack_qwen3_last_layer_bidirectional(model)

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model1 = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B')
model2 = hack_qwen3_last_layer_bidirectional(model=model1)


Layer 27 is_causal set to False


In [27]:
model2

Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM